# Module P07 — Console J1 : saisie, validation et exécution

**Objectif :** coder `phase_j1(etat)` — le pipeline complet :
lire une saisie clavier → parser → valider → exécuter.

> Compatible Google Colab — aucun fichier externe requis.
> Les tests utilisent des saisies simulées (pas de `input()` interactif).

## Section 1 — Rappels synthétiques

### Ce qu'on a vu dans P05/P06 (prérequis)
| Fonction | Rôle |
|----------|------|
| `valider_mouvement(etat, id, col, lig)` | retourne `(bool, str)` |
| `executer_mouvement(etat, id, col, lig)` | modifie l'état, retourne log |

### Pipeline console J1
```
saisie : 'D1 B4'
  1. split() → ['D1', 'B4']
  2. position_depuis_chaine('B4') → (col=1, lig=3)
  3. valider_mouvement(etat, 'D1', 1, 3) → (True, '...')
  4. executer_mouvement(etat, 'D1', 1, 3) → log
```

### Conversion position humaine → indices
| Saisie | col | lig |
|--------|-----|-----|
| 'A1' | 0 | 0 |
| 'B4' | 1 | 3 |
| 'L12' | 11 | 11 |


In [ ]:
# === Stubs des fonctions métier (reproduits inline pour Colab) ===
COLS = 'ABCDEFGHIJKL'
MAX_DEPLACEMENTS_J1 = 3

def valider_mouvement(etat, id_drone, col_cible, lig_cible):
    """Stub simplifié pour les tests P07."""    drone = etat['drones'].get(id_drone)
    if drone is None:
        return False, f'{id_drone} inconnu'
    if drone['hors_service']:
        return False, f'{id_drone} hors service'
    if drone['bloque'] > 0:
        return False, f'{id_drone} bloqué'
    if not (0 <= col_cible < 12 and 0 <= lig_cible < 12):
        return False, 'hors grille'
    return True, 'ok'

def executer_mouvement(etat, id_drone, col_cible, lig_cible):
    """Stub simplifié pour les tests P07."""    drone = etat['drones'][id_drone]
    etat['grille'][drone['lig']][drone['col']] = '.'
    drone['col'], drone['lig'] = col_cible, lig_cible
    drone['batterie'] -= 1
    etat['grille'][lig_cible][col_cible] = id_drone
    return f'{id_drone} → ({col_cible},{lig_cible}), batterie={drone["batterie"]}'

def creer_etat_test():
    grille = [['.' for _ in range(12)] for _ in range(12)]
    etat = {
        'tour': 1, 'score': 0, 'partie_finie': False, 'victoire': False,
        'grille': grille, 'hopital': (0, 0), 'batiments': [],
        'drones': {
            'D1': {'id': 'D1', 'col': 3, 'lig': 3, 'batterie': 15,
                   'batterie_max': 20, 'survivant': None,
                   'bloque': 0, 'hors_service': False}
        },
        'tempetes': {}, 'survivants': {}, 'zones_x': set(), 'historique': []
    }
    etat['grille'][0][0] = 'H'
    etat['grille'][3][3] = 'D1'
    return etat

print('Stubs chargés ✓')

## Section 2 — Exercice guidé

### Étape 1 — Parser : convertir une position texte en indices

In [ ]:
def position_depuis_chaine(chaine):
    """Convertit 'B4' en (col=1, lig=3). Retourne (None, None) si invalide."""    # TODO 1 : extraire la lettre (chaine[0]) et le numéro (chaine[1:])
    # TODO 2 : trouver l'index de la lettre dans COLS (0 à 11)
    # TODO 3 : convertir le numéro en int et soustraire 1 (1-12 → 0-11)
    # TODO 4 : vérifier que col et lig sont dans [0, 11]
    # Utiliser try/except ValueError pour gérer int() qui échoue
    try:
        pass  # à remplacer
    except (ValueError, IndexError):
        pass
    return None, None

In [ ]:
# Vérification du parser
assert position_depuis_chaine('A1') == (0, 0), f"A1 → attendu (0,0)"assert position_depuis_chaine('B4') == (1, 3), f"B4 → attendu (1,3)"assert position_depuis_chaine('L12') == (11, 11), f"L12 → attendu (11,11)"assert position_depuis_chaine('Z9') == (None, None), 'lettre invalide'
assert position_depuis_chaine('B99') == (None, None), 'numéro trop grand'
assert position_depuis_chaine('') == (None, None), 'chaîne vide'
print('✓ Étape 1 validée — parser OK')

### Étape 2 — Boucle J1 avec saisie simulée
Pour tester sans `input()` interactif, on simule les saisies avec un "injecteur" :
```python
saisies = iter(['D1 D4', 'D1 D5', 'fin'])
def input_simule(prompt=''):
    return next(saisies)
```

In [ ]:
def phase_j1_v1(etat, input_fn=input):
    """Phase J1 — version avec input_fn injectable pour les tests."""    nb_depl = 0
    logs = []

    while nb_depl < MAX_DEPLACEMENTS_J1:
        saisie = input_fn(f'J1 [{nb_depl}/{MAX_DEPLACEMENTS_J1}] > ').strip()

        # TODO 1 : si saisie est 'fin', 'passe' ou vide → break
        # if saisie.lower() in (...):
        #     break

        # TODO 2 : découper avec split() et vérifier len == 2
        parties = saisie.split()
        # if len(parties) != 2:
        #     print('Format : D1 B4')
        #     continue

        id_drone = parties[0].upper() if parties else ''
        pos_str = parties[1] if len(parties) > 1 else ''

        # TODO 3 : vérifier que id_drone est dans etat['drones']
        # if id_drone not in etat['drones']:
        #     print(f'Drone {id_drone} inconnu')
        #     continue

        col, lig = position_depuis_chaine(pos_str)
        if col is None:
            print(f"Position '{pos_str}' invalide")
            continue

        ok, msg = valider_mouvement(etat, id_drone, col, lig)
        if not ok:
            print(f'Mouvement refusé : {msg}')
            continue

        log = executer_mouvement(etat, id_drone, col, lig)
        logs.append(log)
        # TODO 4 : incrémenter nb_depl
        # nb_depl += 1

    return logs

In [ ]:
# Test : 2 déplacements puis 'fin'
etat = creer_etat_test()
saisies = iter(['D1 D4', 'D1 D5', 'fin'])
logs = phase_j1_v1(etat, input_fn=lambda p='': next(saisies))

print('Logs :', logs)
assert len(logs) == 2, f'Attendu 2 déplacements, obtenu {len(logs)}'
assert etat['drones']['D1']['lig'] == 4, 'D1 doit être en lig=4 (D5)'
print('✓ Étape 2 validée')

# Test : 3 déplacements = limite atteinte (pas de 4e)
etat2 = creer_etat_test()
saisies2 = iter(['D1 D4', 'D1 D5', 'D1 D6', 'D1 D7'])  # 4e ignorable
logs2 = phase_j1_v1(etat2, input_fn=lambda p='': next(saisies2))
assert len(logs2) == 3, f'Attendu max 3 déplacements, obtenu {len(logs2)}'
print('✓ Limite 3 déplacements respectée')

### Étape 3 — Robustesse : saisies mal formées
La boucle doit absorber les erreurs sans crasher et redemander la saisie.

In [ ]:
# Test robustesse : saisies invalides absorbées
etat3 = creer_etat_test()
# Saisies : 1 invalide (format), 1 invalide (drone inconnu), 1 valide, fin
saisies3 = iter(['D1', 'D9 B4', 'D1 D4', 'fin'])
logs3 = phase_j1_v1(etat3, input_fn=lambda p='': next(saisies3))

assert len(logs3) == 1, f'1 seul mouvement valide attendu, obtenu {len(logs3)}'
print('✓ Étape 3 validée — robustesse OK')
print('\n✓ Module P07 complet !')